### Step 1: Install Required Libraries
Installing the necessary packages for our RAG pipeline, including LlamaIndex, Cohere (for embeddings), and Pinecone (for vector storage).

In [71]:
%pip install llama-index llama-index-llms-cohere llama-index-embeddings-cohere llama-index-vector-stores-pinecone python-dotenv pinecone-client

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [72]:
%pip install pip-system-certs

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### Step 2: Load Environment Variables
Securely loading the API keys for Cohere and Pinecone from the `.env` file to authenticate our requests.

In [73]:
import os
from dotenv import load_dotenv

# טעינת המפתחות
load_dotenv()

cohere_key = os.environ.get("COHERE_API_KEY")
pinecone_key = os.environ.get("PINECONE_API_KEY")

if cohere_key and pinecone_key:
    print("איזה יופי! המפתחות נטענו בהצלחה מקובץ ה-.env 🎉")
else:
    print("שגיאה: חסר מפתח. ודאי שקובץ ה-.env שמור באותה תיקייה וכתוב נכון.")

איזה יופי! המפתחות נטענו בהצלחה מקובץ ה-.env 🎉


### Step 3: Load Documents and Assign Metadata
Reading all Markdown files from the data directory. We assign metadata (tool name) based on the folder structure to clearly separate documents generated by Cursor from those generated by Kiro, as required by the architecture.

In [74]:
from llama_index.core import SimpleDirectoryReader
import os

# הפונקציה שמוסיפה מטא-דאטה לפי שם התיקייה
def get_meta(file_path):
    if "cursor_docs" in file_path:
        return {"tool": "Cursor", "file_name": os.path.basename(file_path)}
    elif "kiro_docs" in file_path:
        return {"tool": "Kiro", "file_name": os.path.basename(file_path)}
    return {"tool": "General", "file_name": os.path.basename(file_path)}

print("מתחילה לקרוא את הקבצים...")

# טעינת הקבצים מהתיקייה
loader = SimpleDirectoryReader(
    input_dir="./data", 
    recursive=True,
    file_metadata=get_meta
)

documents = loader.load_data()

print(f"הצלחה! נטענו {len(documents)} מסמכים.")
if len(documents) > 0:
    print("דוגמה למטא-דאטה של המסמך הראשון:", documents[0].metadata)

מתחילה לקרוא את הקבצים...
הצלחה! נטענו 10 מסמכים.
דוגמה למטא-דאטה של המסמך הראשון: {'tool': 'Cursor', 'file_name': 'database_schema.md'}


### Step 4: Document Chunking
Parsing the loaded Markdown documents into smaller, semantic nodes (chunks) using LlamaIndex's `MarkdownNodeParser`. This splits the text logically based on Markdown headers, which significantly improves retrieval accuracy.

In [75]:
from llama_index.core.node_parser import MarkdownNodeParser

print("Starting to chunk documents...")

# Define the parser
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)

print(f"Success! {len(documents)} documents were parsed into {len(nodes)} nodes (chunks).")

Starting to chunk documents...
Success! 10 documents were parsed into 256 nodes (chunks).


### Step 5: Create Embeddings and Store in Pinecone
Initializing the Cohere embedding model (`embed-multilingual-v3.0`) to convert our text chunks into vectors. Then, we connect to the Pinecone vector database, create an index if it doesn't exist, and store all the vectorized nodes.

In [76]:
import logging
import os

# משתיקים את הלוגים כדי שהמחברת לא תקטע את הפלט בטעות
logging.getLogger("httpx").setLevel(logging.WARNING)

from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings, VectorStoreIndex, StorageContext
from llama_index.vector_stores.pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

print("Configuring Cohere Embedding model...")
cohere_api_key = os.environ.get("COHERE_API_KEY")

# הוספנו את השורה embed_batch_size כדי למנוע את החסימה של קוהיר!
embed_model = CohereEmbedding(
    api_key=cohere_api_key,
    model_name="embed-multilingual-v3.0",
    input_type="search_document",
    embed_batch_size=90 
)
Settings.embed_model = embed_model

print("Connecting to Pinecone...")
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index_name = "rag-agent-index"

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    print(f"Creating new Pinecone index '{index_name}' (this may take a minute)...")
    pc.create_index(
        name=index_name,
        dimension=1024, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

pinecone_index = pc.Index(index_name)
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("Embedding nodes and saving to Pinecone (please wait)...")
# כאן המערכת תשלח את המידע לקוהיר בחבילות גדולות ובטוחות
index = VectorStoreIndex(
    nodes, 
    storage_context=storage_context
)

print("🎉 All done! Data is successfully indexed in Pinecone.")

Configuring Cohere Embedding model...
Connecting to Pinecone...
Embedding nodes and saving to Pinecone (please wait)...


Upserted vectors:   0%|          | 0/256 [00:00<?, ?it/s]

2026-02-21 22:43:47,251 - WARNING - Retrying (JitterRetry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2406)')': /vectors/upsert


KeyboardInterrupt: 

### Step 6: Setup LLM and Query Engine
Initializing the LLM (OpenAI's GPT model) to synthesize the final answers. We then create a `QueryEngine` from our Pinecone index to fetch the most relevant chunks and formulate a human-readable answer based on our project documents.

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from dotenv import load_dotenv
import os

print("טוענת את המפתחות מחדש...")
load_dotenv(override=True) 

print("מגדירה את מודל השפה (LLM) של OpenAI...")

openai_key = os.environ.get("OPENAI_API_KEY")
if not openai_key:
    print("❌ שגיאה: המפתח של OpenAI עדיין חסר! ודאי ששמרת את קובץ ה-.env")
else:
    print("✅ מפתח OpenAI זוהה בהצלחה!")


llm = OpenAI(model="gpt-4o-mini", temperature=0.3, api_key=openai_key)
Settings.llm = llm

print("בונה את מנוע החיפוש והתשאול (Query Engine)...")
query_engine = index.as_query_engine(
    similarity_top_k=3
)

print("מנוע החיפוש מוכן! בואי נעשה בדיקת חיבור קטנה...")

response = query_engine.query("What is the main purpose of this project?")
print("\nתשובת המערכת:")
print(response)

טוענת את המפתחות מחדש...
מגדירה את מודל השפה (LLM) של OpenAI...
✅ מפתח OpenAI זוהה בהצלחה!
בונה את מנוע החיפוש והתשאול (Query Engine)...
מנוע החיפוש מוכן! בואי נעשה בדיקת חיבור קטנה...

תשובת המערכת:
The main purpose of this project is to develop a Project Management tool that enables users to create, manage, and track projects associated with specific clients, including functionalities for project status updates, editing, deletion, and data persistence.


In [ ]:
import gradio as gr

# הפונקציה שמקשרת בין הצ'אט למנוע החיפוש שלנו
def chat_with_rag(message, history):
    try:
        # שליחת השאלה מהצ'אט ל-RAG
        response = query_engine.query(message)
        return str(response)
    except Exception as e:
        return f"אופס, משהו השתבש: {str(e)}"

print("מכינה את ממשק הצ'אט...")

# עיצוב והגדרת ממשק הצ'אט
demo = gr.ChatInterface(
    fn=chat_with_rag,
    title="🤖 FreelanceFlow RAG Agent",
    description="שאלי אותי כל שאלה על קבצי האפיון, הארכיטקטורה, והחלטות הפיתוח של הפרויקט!",
    examples=[
        "What is the database schema?",
        "What are the main coding rules?",
        "Why did we choose this architecture?"
    ]
)

print("מפעילה את הצ'אט! 🚀")
# share=True ייצור לך קישור ציבורי שאפשר לשתף
demo.launch(share=True)

C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


מכינה את ממשק הצ'אט...
מפעילה את הצ'אט! 🚀
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://384183f62e971c62db.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Step 8: Upgrading to an Event-Driven Architecture (Workflow)
In this step, we transition from a standard query engine to a structured, event-driven `Workflow` using LlamaIndex. 
This architecture breaks down the RAG pipeline into distinct, manageable steps (Validation -> Retrieval -> Generation).
It introduces strict validations to prevent unnecessary LLM calls (e.g., empty queries, missing context, or low confidence scores) and utilizes a `Context` object for state management across events.

In [ ]:
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
from llama_index.core import Settings

# 1. הגדרת האירועים (Events) שמקשרים בין השלבים
class RetrievalEvent(Event):
    query: str

class GenerationEvent(Event):
    nodes: list
    query: str  # 🟢 הפתרון האלגנטי: הוספנו את השאלה לכאן כדי שהיא "תרכב" על האירוע

# 2. בניית ה-Workflow מונחה האירועים
class RAGWorkflow(Workflow):
    def __init__(self, index, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # הופכים את האינדקס לשולפן בלבד
        self.retriever = index.as_retriever(similarity_top_k=3)

    @step
    async def validate_input(self, ev: StartEvent) -> RetrievalEvent | StopEvent:
        """Step 1: Input Validation"""
        query = ev.get("query")
        
        # ולידציה 1: קלט חסר או קצר מדי
        if not query or len(query.strip()) < 3:
            return StopEvent(result="Validation Error: Input is empty or too short. Please ask a clear question.")
        
        print("✅ Step 1: Input validated successfully.")
        return RetrievalEvent(query=query)

    @step
    async def retrieve_data(self, ev: RetrievalEvent) -> GenerationEvent | StopEvent:
        """Step 2: Data Retrieval & Context Validation"""
        nodes = self.retriever.retrieve(ev.query)
        
        # ולידציה 2: לא נמצאו נתונים
        if not nodes:
            return StopEvent(result="Validation Error: No relevant context found for this query.")
        
        # ולידציה 3: רמת ביטחון נמוכה של החיפוש
        highest_score = nodes[0].score if nodes[0].score is not None else 1.0
        if highest_score < 0.2:
            return StopEvent(result="Validation Error: Confidence score too low. Please be more specific.")

        print(f"✅ Step 2: Retrieved {len(nodes)} relevant documents.")
        # 🟢 אנחנו אורזים את השאלה יחד עם התוצאות ושולחים לשלב הבא
        return GenerationEvent(nodes=nodes, query=ev.query)

    @step
    async def generate_response(self, ev: GenerationEvent) -> StopEvent:
        """Step 3: Answer Generation via LLM"""
        # 🟢 שולפים את השאלה ישירות מתוך האירוע, בלי לחפש בזיכרון גלובלי
        query = ev.query
        
        # חיבור כל הטקסטים שנמצאו למחרוזת אחת
        context_str = "\n\n".join([n.get_content() for n in ev.nodes])
        
        # יצירת הפרומפט הסופי
        prompt = f"Based on the following information:\n{context_str}\n\nPlease answer the question: {query}\nAnswer in Hebrew."
        
        # קריאה ל-LLM
        response = Settings.llm.complete(prompt)
        print("✅ Step 3: Response generated successfully.")
        
        return StopEvent(result=str(response))


# ==========================================
# אזור הבדיקות (טסטים) להרצה במחברת
# ==========================================
print("מפעילה את ה-Workflow ובודקת אותו...\n")

# יצירת מופע של ה-Workflow שלנו
workflow = RAGWorkflow(index=index, timeout=60.0)

# טסט 1: שאלה תקינה לחלוטין (אמורה לעבור את כל ה-Steps)
print("--- Test 1: Valid Query ---")
valid_result = await workflow.run(query="מה הפרויקט עושה?")
print("System Answer:", valid_result, "\n")

# טסט 2: קלט שגוי/קצר מדי (אמור לעצור מיד ב-Step 1 ולחסוך קריאה ל-LLM)
print("--- Test 2: Invalid Query (Too short) ---")
invalid_result = await workflow.run(query="א")
print("System Answer:", invalid_result)

מפעילה את ה-Workflow ובודקת אותו...

--- Test 1: Valid Query ---
✅ Step 1: Input validated successfully.
✅ Step 2: Retrieved 1 relevant documents.
✅ Step 3: Response generated successfully.
System Answer: הפרויקט מנהל את תהליך ניהול הפרויקטים עבור לקוחות. הוא מאפשר יצירת פרויקטים שמקושרים ללקוח ספציפי, קביעת מצב הפרויקט (כגון "לא התחיל", "בתהליך", או "הושלם"), עריכת פרטי הפרויקט, ומחיקת פרויקטים כולל כל יומני הזמן הקשורים אליהם. בנוסף, הפרויקט מציג מידע על הלקוח הקשור לכל פרויקט ושומר את כל הנתונים המקומיים בזיכרון. 

--- Test 2: Invalid Query (Too short) ---
System Answer: Validation Error: Input is empty or too short. Please ask a clear question.


In [ ]:
import json
from pydantic import BaseModel, Field
from typing import List
from llama_index.core import SimpleDirectoryReader, Settings, PromptTemplate

# 1. הגדרת הסכמה (Schema) באמצעות Pydantic
class Decision(BaseModel):
    id: str = Field(description="מזהה ייחודי להחלטה, למשל dec-001")
    title: str = Field(description="כותרת קצרה להחלטה (עד 4 מילים)")
    summary: str = Field(description="סיכום ברור של ההחלטה שהתקבלה")
    tags: List[str] = Field(description="תגיות נושא, למשל: db, architecture, frontend")

class Rule(BaseModel):
    id: str = Field(description="מזהה ייחודי לכלל, למשל rule-001")
    rule: str = Field(description="ההנחיה או הכלל שנקבע בפרויקט")
    scope: str = Field(description="התחום עליו הכלל חל, למשל ui, backend, auth")

class WarningItem(BaseModel):
    id: str = Field(description="מזהה ייחודי לאזהרה, למשל warn-001")
    area: str = Field(description="האזור הרגיש אליו מתייחסת האזהרה בקוד")
    message: str = Field(description="תוכן האזהרה או מה אסור לעשות")
    severity: str = Field(description="רמת חומרה: high, medium, low")

class ExtractedData(BaseModel):
    decisions: List[Decision] = Field(default_factory=list)
    rules: List[Rule] = Field(default_factory=list)
    warnings: List[WarningItem] = Field(default_factory=list)

print("📝 הסכמה הוגדרה בהצלחה! מתחילה בסריקת הקבצים (כולל תיקיות משנה)...")

# 2. קריאת קובצי התיעוד מתיקיית הנתונים וכל תתי-התיקיות (cursor, kiro וכו')
documents = SimpleDirectoryReader("data", recursive=True).load_data()
full_text = "\n\n".join([doc.text for doc in documents])

# 3. בניית הפרומפט לחילוץ בעזרת PromptTemplate
# 🟢 התיקון כאן: עטפנו את הטקסט ב-PromptTemplate
prompt_template = PromptTemplate(f"""
אתה מנתח מערכות מקצועי. קרא את מסמכי האפיון הבאים של הפרויקט.
עליך לחלץ מתוכם את כל:
1. ההחלטות הטכניות או העסקיות (Decisions).
2. הכללים, חוקי הפיתוח וההנחיות (Rules).
3. האזהרות והדברים שסומנו כרגישים או מסוכנים (Warnings).

החזר את הנתונים במבנה JSON התואם לסכמה שהוגדרה מראש.

טקסט המסמכים המלא:
{full_text}
""")

# 4. ביצוע החילוץ המובנה בעזרת ה-LLM
# 🟢 התיקון כאן: שינינו את שם הפרמטר ל-prompt והעברנו את האובייקט שיצרנו
extracted_pydantic_obj = Settings.llm.structured_predict(
    ExtractedData, 
    prompt=prompt_template
)

# 5. המרה למילון ושמירה לקובץ JSON
output_dict = extracted_pydantic_obj.model_dump()

output_filename = "project_structured_data.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(output_dict, f, indent=4, ensure_ascii=False)

print(f"✅ איזה יופי! הנתונים חולצו בהצלחה ונשמרו לקובץ: {output_filename}")
print("-" * 50)
print("הצצה קטנה לנתונים שחולצו:")
print(json.dumps(output_dict, indent=2, ensure_ascii=False)[:600] + "\n... (המשך הנתונים בקובץ)")

📝 הסכמה הוגדרה בהצלחה! מתחילה בסריקת הקבצים (כולל תיקיות משנה)...
✅ איזה יופי! הנתונים חולצו בהצלחה ונשמרו לקובץ: project_structured_data.json
--------------------------------------------------
הצצה קטנה לנתונים שחולצו:
{
  "decisions": [
    {
      "id": "dec-001",
      "title": "Invoice Data Model and TimeLog Linking",
      "summary": "Add an Invoice entity with status (Draft, Sent, Paid), linked to TimeLogs via `invoiceId` on TimeLog and `timeLogIds` on Invoice. When an invoice is generated, update TimeLogs to set `invoiceId` (marking them as billed).",
      "tags": [
        "db",
        "architecture",
        "frontend"
      ]
    },
    {
      "id": "dec-002",
      "title": "Pending Invoices Definition for Dashboard Widget",
      "summary": "\"Pending Invoices\" = invoices with status Draft or
... (המשך הנתונים בקובץ)


In [ ]:
import json
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step, Context
from llama_index.core import Settings

# 1. הגדרת אירועים חדשים לניתוב
class RetrievalEvent(Event):
    query: str

class StructuredDataEvent(Event):
    query: str
    category: str # decisions, rules, or warnings

class GenerationEvent(Event):
    nodes_text: str
    query: str

# 2. ה-Workflow החכם עם הראוטר
class AgenticRAGWorkflow(Workflow):
    def __init__(self, index, structured_data_path, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.retriever = index.as_retriever(similarity_top_k=3)
        with open(structured_data_path, "r", encoding="utf-8") as f:
            self.structured_data = json.load(f)

    @step
    async def router(self, ev: StartEvent) -> RetrievalEvent | StructuredDataEvent | StopEvent:
        """שלב הניתוב: מחליט לאן ללכת לפי סוג השאלה"""
        query = ev.get("query")
        if not query:
            return StopEvent(result="Please ask a question.")

        # פרומפט קצר ל-LLM שיחליט על הניתוב
        choices = [
            "STRUCTURED_DATA: לשאלות על רשימות, חוקים, החלטות טכניות, אזהרות או הנחיות ספציפיות.",
            "VECTOR_SEARCH: לשאלות כלליות, הסברים על מהות הפרויקט או נושאים שדורשים הבנה רחבה."
        ]
        
        router_prompt = f"Given the query: '{query}', which tool is better? {choices}. Reply ONLY with the tool name and if it is STRUCTURED_DATA, add the category (decisions/rules/warnings) after a comma."
        response = str(Settings.llm.complete(router_prompt)).strip()

        print(f"🤖 Router decision: {response}")

        if "STRUCTURED_DATA" in response:
            category = response.split(",")[1].strip().lower() if "," in response else "decisions"
            return StructuredDataEvent(query=query, category=category)
        else:
            return RetrievalEvent(query=query)

    @step
    async def retrieve_vector_data(self, ev: RetrievalEvent) -> GenerationEvent:
        """מסלול א': חיפוש סמנטי רגיל"""
        nodes = self.retriever.retrieve(ev.query)
        context_str = "\n\n".join([n.get_content() for n in nodes])
        print("🔍 Path selected: Vector Search")
        return GenerationEvent(nodes_text=context_str, query=ev.query)

    @step
    async def retrieve_structured_data(self, ev: StructuredDataEvent) -> GenerationEvent:
        """מסלול ב': שליפה מה-JSON המובנה"""
        category = ev.category if ev.category in self.structured_data else "decisions"
        data_items = self.structured_data.get(category, [])
        
        # הופכים את ה-JSON לטקסט שה-LLM יוכל לעבד
        context_str = f"Found the following {category} in structured data:\n" + json.dumps(data_items, indent=2, ensure_ascii=False)
        print(f"📊 Path selected: Structured Data ({category})")
        return GenerationEvent(nodes_text=context_str, query=ev.query)

    @step
    async def generate_response(self, ev: GenerationEvent) -> StopEvent:
        """שלב סופי: יצירת תשובה על בסיס המידע שנבחר"""
        prompt = f"Based on this info:\n{ev.nodes_text}\nAnswer the query: {ev.query}\nAnswer in Hebrew."
        response = Settings.llm.complete(prompt)
        return StopEvent(result=str(response))

# הרצה ובדיקה
workflow = AgenticRAGWorkflow(index=index, structured_data_path="project_structured_data.json", timeout=60.0)

print("--- בדיקה 1: שאלה כללית (אמורה ללכת לווקטורים) ---")
res1 = await workflow.run(query="מה המטרה הכללית של הפרויקט?")
print(f"תשובה: {res1}\n")

print("--- בדיקה 2: שאלה על החלטות (אמורה ללכת ל-JSON) ---")
res2 = await workflow.run(query="אילו החלטות טכניות התקבלו לגבי חשבוניות?")
print(f"תשובה: {res2}")

--- בדיקה 1: שאלה כללית (אמורה ללכת לווקטורים) ---
🤖 Router decision: VECTOR_SEARCH
🔍 Path selected: Vector Search
תשובה: המטרה הכללית של הפרויקט היא לפתח מערכת ניהול פרויקטים שתאפשר למנהל הפרויקטים ליצור, לערוך, למחוק ולנהל פרויקטים הקשורים ללקוחות ספציפיים, תוך שמירה על סטטוס הפרויקטים ונתוני זמן קשורים.

--- בדיקה 2: שאלה על החלטות (אמורה ללכת ל-JSON) ---
🤖 Router decision: STRUCTURED_DATA, decisions
📊 Path selected: Structured Data (decisions)
תשובה: ההחלטות הטכניות שהתקבלו לגבי חשבוניות הן:

1. **מודל נתוני חשבונית וקישור ל-TimeLog**: הוחלט להוסיף ישות חשבונית עם מצב (טיוטה, נשלחה, שולם), המקושרת ל-TimeLogs דרך `invoiceId` ב-TimeLog ו-`timeLogIds` בחשבונית. כאשר נוצרה חשבונית, יש לעדכן את ה-TimeLogs כדי לקבוע את `invoiceId` (ולסמן אותם כחשבוניות שנשלחו).

2. **הגדרת חשבוניות ממתינות עבור ווידג'ט בלוח המחוונים**: הוחלט ש"חשבוניות ממתינות" הן חשבוניות עם מצב טיוטה או נשלחה (כלומר, לא שולם עדיין). הווידג'ט מציג את מספר החשבוניות הממתינות ואת הסכום הכולל שלהן.


In [ ]:
import gradio as gr

# הפונקציה שמקשרת בין הצ'אט ל-Workflow החכם שלנו
async def chat_with_rag(message, history):
    try:
        # 🟢 שינוי חשוב: אנחנו מריצים את ה-workflow ולא את ה-query_engine
        # message היא השאלה מהמשתמש
        response = await workflow.run(query=message)
        
        return str(response)
    except Exception as e:
        return f"אופס, משהו השתבש: {str(e)}"

print("מכינה את ממשק הצ'אט ה-Agentic...")

# עיצוב והגדרת ממשק הצ'אט
demo = gr.ChatInterface(
    fn=chat_with_rag,
    title="🤖 FreelanceFlow AI Agent",
    description="שלום! אני הסוכן החכם של הפרויקט. אני יודע לחפש בתיעוד הכללי או לשלוף החלטות וחוקים מדויקים מה-JSON.",
    examples=[
        "מה המטרה של הפרויקט?",
        "אילו החלטות טכניות התקבלו?",
        "מהם חוקי ה-UI שנקבעו?",
        "האם יש אזהרות לגבי ה-JWT?"
    ],
    theme="soft" # עיצוב נקי ונעים
)

print("מפעילה את הצ'אט! 🚀")
# share=True יאפשר למורה שלך להיכנס לקישור מהמחשב שלה (למשך 72 שעות)
demo.launch(share=True)

C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


מכינה את ממשק הצ'אט ה-Agentic...
מפעילה את הצ'אט! 🚀
* Running on local URL:  http://127.0.0.1:7863

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/02/21 22:24:45 [W] [service.go:132] login to server failed: session shutdown


🤖 Router decision: STRUCTURED_DATA, decisions
📊 Path selected: Structured Data (decisions)


In [ ]:
# 1. התקנת הספרייה הנדרשת (הריצי בתא נפרד אם צריך)
%pip install pyvis

# ניסיון תיקון לשיטה האוטומטית
try:
    from llama_index.core.workflow import draw_all_possible_flows
    draw_all_possible_flows(AgenticRAGWorkflow, filename="workflow_graph.html")
    print("הקובץ workflow_graph.html נוצר בהצלחה! פתחי אותו בדפדפן.")
except Exception as e:
    print(f"עדיין יש שגיאה: {e}. מומלץ לעבור לשיטה 2 המקצועית.")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
עדיין יש שגיאה: cannot import name 'draw_all_possible_flows' from 'llama_index.core.workflow' (C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\llama_index\core\workflow\__init__.py). מומלץ לעבור לשיטה 2 המקצועית.
